# Counterexamples to strong I-contractibility

First run `python3 tools/copy_notebook.py` from the repository root, then open the copy under `results/notebook/`. Jupyter may save outputs there without changing the distributed notebook. The Python cells use only the standard library and preserve the manuscript labels.


In [ ]:
from pathlib import Path
import csv, json, subprocess, sys
sys.dont_write_bytecode = True
start = Path.cwd().resolve()
root = next((p for p in (start, *start.parents) if (p / "generate.py").is_file() and (p / "finite/certificates").is_dir()), None)
if root is None:
    raise RuntimeError("Open the notebook copy under this repository's results directory")
sys.path.insert(0, str(root / "finite/certificates"))
from finite_checks import decode, strong, h1
def export(name, extension=False):
    command = [sys.executable, str(root / "generate.py"), "--graph", name, "--format", "json"]
    if extension:
        command.append("--extension")
    return json.loads(subprocess.check_output(command, text=True))


## A 13-vertex graph with odd torsion

The clique complex of $G_{13}$ has $H_1=\mathbb Z/3$. Its disc presentation and the proof of mod-2 acyclicity appear in the article. Here we recompute the integral first homology from the triangle relations.


In [ ]:
g13 = export("G13")
assert (g13["n"], g13["m"]) == (13, 40)
assert h1(decode(g13["graph6"])) == (0, (3,))
print("G13:", g13["n"], "vertices,", g13["m"], "edges; H1 = Z/3")


## One edge changes strong contractibility

The 44-edge graph $G_{14}^a$ is not strong I-contractible. Adding $\{0,1\}$ gives a strong graph. The checked common neighbourhood makes the reverse deletion admissible, so this pair refutes Conjecture 15.


In [ ]:
ga = export("G14a")
plus = export("G14a", extension=True)
assert (ga["n"], ga["m"], plus["m"]) == (14, 44, 45)
assert not strong(decode(ga["graph6"]))
assert strong(decode(plus["graph6"]))
print("G14a is non-strong; its one-edge extension is strong.")


## The two coefficient columns

Every entry in row 14 counts connected $K_4$-free graphs up to isomorphism. Rows 3–13 include all connected graphs. Reading the table does not regenerate those universes.


In [ ]:
with (root / "data/cardinalities.csv").open(newline="") as stream:
    rows = list(csv.DictReader(stream))
for row in rows[-2:]:
    gap = int(row["acyclic_F2"]) - int(row["acyclic_Z"])
    assert gap == (3 if row["n"] == "13" else 625)
    print("n =", row["n"], "universe =", row["universe"], "F2/Z gap =", gap)


## Verification and proofs

Run `python3 reproduce.py` from the repository root for the required finite checks and a saved report. `python3 reproduce.py --extended` also compares the current C recurrences on every labelled graph of order at most five. The mathematical supplement is in `supplement/main.pdf`; the main article is in `paper/main.pdf`. The full enumeration route is documented separately in `docs/full-reproduction.md`.
